In [2]:
!nvcc --version || true

import numba
print("numba:", numba.__version__)

from numba import cuda
cuda.cudadrv.libs.test()

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
numba: 0.60.0
Finding driver from candidates:
	libcuda.so
	libcuda.so.1
	/usr/lib/libcuda.so
	/usr/lib/libcuda.so.1
	/usr/lib64/libcuda.so
	/usr/lib64/libcuda.so.1
Using loader <class 'ctypes.CDLL'>
	Trying to load driver...	ok
		Loaded from libcuda.so
	Mapped libcuda.so paths:
		/usr/lib64-nvidia/libcuda.so.580.82.07
Finding nvvm from NVIDIA NVCC Wheel
	Located at /usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvcc/nvvm/lib64/libnvvm.so
	Trying to open library...	ok
Finding nvrtc from NVIDIA NVCC Wheel
	Located at /usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12
	Trying to open library...	ok
Finding cudadevrt from System
	Located at /usr/local/cuda/lib64/libcudadevrt.a
	Checking library...	ok
Finding libdevice from NVIDIA NVCC Wheel
	Located at /us

True

In [1]:
import os
os.environ["NUMBA_FORCE_CUDA_CC"] = "9.0"

In [14]:
# BlackwellでNumbaが compute_120 エラーを出す場合は、
# from numba import cuda より前にこれを入れる
import os
os.environ.setdefault("NUMBA_FORCE_CUDA_CC", "9.0")

import hashlib
import time

import numpy as np
from numba import cuda, uint8, uint32, uint64, int32


K_HOST = np.array([
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
], dtype=np.uint32)


MAX_THREADS_PER_BLOCK = 1024
UINT64_MAX = (1 << 64) - 1


@cuda.jit(device=True)
def rotr(x, n):
    return uint32((x >> n) | (x << (32 - n)))


@cuda.jit(device=True)
def sha256_one_block(block, digest, K):
    w = cuda.local.array(64, dtype=uint32)

    for i in range(16):
        j = i * 4
        w[i] = uint32(
            (uint32(block[j]) << 24)
            | (uint32(block[j + 1]) << 16)
            | (uint32(block[j + 2]) << 8)
            | uint32(block[j + 3])
        )

    for i in range(16, 64):
        s0 = rotr(w[i - 15], 7) ^ rotr(w[i - 15], 18) ^ (w[i - 15] >> 3)
        s1 = rotr(w[i - 2], 17) ^ rotr(w[i - 2], 19) ^ (w[i - 2] >> 10)
        w[i] = uint32(w[i - 16] + s0 + w[i - 7] + s1)

    a = uint32(0x6a09e667)
    b = uint32(0xbb67ae85)
    c = uint32(0x3c6ef372)
    d = uint32(0xa54ff53a)
    e = uint32(0x510e527f)
    f = uint32(0x9b05688c)
    g = uint32(0x1f83d9ab)
    h = uint32(0x5be0cd19)

    for i in range(64):
        S1 = rotr(e, 6) ^ rotr(e, 11) ^ rotr(e, 25)
        ch = (e & f) ^ ((e ^ uint32(0xffffffff)) & g)
        temp1 = uint32(h + S1 + ch + K[i] + w[i])

        S0 = rotr(a, 2) ^ rotr(a, 13) ^ rotr(a, 22)
        maj = (a & b) ^ (a & c) ^ (b & c)
        temp2 = uint32(S0 + maj)

        h = g
        g = f
        f = e
        e = uint32(d + temp1)
        d = c
        c = b
        b = a
        a = uint32(temp1 + temp2)

    digest[0] = uint32(a + uint32(0x6a09e667))
    digest[1] = uint32(b + uint32(0xbb67ae85))
    digest[2] = uint32(c + uint32(0x3c6ef372))
    digest[3] = uint32(d + uint32(0xa54ff53a))
    digest[4] = uint32(e + uint32(0x510e527f))
    digest[5] = uint32(f + uint32(0x9b05688c))
    digest[6] = uint32(g + uint32(0x1f83d9ab))
    digest[7] = uint32(h + uint32(0x5be0cd19))


@cuda.jit(device=True)
def count_leading_f_nibbles(digest):
    count = 0

    for i in range(8):
        word = digest[i]

        for j in range(8):
            shift = 28 - j * 4
            nibble = (word >> shift) & uint32(0xF)

            if nibble == uint32(0xF):
                count += 1
            else:
                return count

    return count


@cuda.jit(device=True)
def write_u64_decimal(msg_block, pos, x):
    # nonce を 10進数文字列として msg_block に書く
    # 例: 12345 -> b"12345"
    if x == uint64(0):
        msg_block[pos] = uint8(48)  # '0'
        return int32(1)

    y = x
    length = int32(0)

    while y > uint64(0):
        length += 1
        y = y // uint64(10)

    y = x
    i = length - 1

    while i >= 0:
        digit = y % uint64(10)
        msg_block[pos + i] = uint8(uint32(48) + uint32(digit))
        y = y // uint64(10)
        i -= 1

    return length


@cuda.jit
def mine_max_f_kernel(
    prefix,
    prefix_len,
    start_nonce,
    block_best_scores,
    block_best_nonces,
    K,
):
    tid = cuda.threadIdx.x
    bid = cuda.blockIdx.x
    idx = cuda.grid(1)

    nonce = uint64(start_nonce) + uint64(idx)

    msg_block = cuda.local.array(64, dtype=uint8)
    digest = cuda.local.array(8, dtype=uint32)

    s_scores = cuda.shared.array(MAX_THREADS_PER_BLOCK, dtype=int32)
    s_nonces = cuda.shared.array(MAX_THREADS_PER_BLOCK, dtype=uint64)

    for i in range(64):
        msg_block[i] = uint8(0)

    # prefix, 例: b"mumumu-"
    for i in range(prefix_len):
        msg_block[i] = prefix[i]

    # ここが重要:
    # nonce を raw 8 bytes ではなく、10進数文字列として後ろに付ける
    # 例: b"mumumu-" + b"12345"
    suffix_len = write_u64_decimal(msg_block, prefix_len, nonce)

    msg_len = prefix_len + suffix_len

    # SHA-256 padding
    msg_block[msg_len] = uint8(0x80)

    bit_len = uint64(msg_len * 8)

    for i in range(8):
        shift = (7 - i) * 8
        msg_block[56 + i] = uint8((bit_len >> shift) & uint64(0xff))

    sha256_one_block(msg_block, digest, K)

    score = count_leading_f_nibbles(digest)

    s_scores[tid] = score
    s_nonces[tid] = nonce

    cuda.syncthreads()

    # block内で最大scoreを探す
    stride = cuda.blockDim.x // 2

    while stride > 0:
        if tid < stride:
            other_score = s_scores[tid + stride]
            other_nonce = s_nonces[tid + stride]

            my_score = s_scores[tid]
            my_nonce = s_nonces[tid]

            if other_score > my_score or (
                other_score == my_score and other_nonce < my_nonce
            ):
                s_scores[tid] = other_score
                s_nonces[tid] = other_nonce

        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        block_best_scores[bid] = s_scores[0]
        block_best_nonces[bid] = s_nonces[0]


def submit_value(prefix: str, nonce: int) -> str:
    # 実際に提出する値
    return f"{prefix}{nonce}"


def submit_bytes(prefix: str, nonce: int) -> bytes:
    # 提出値をUTF-8エンコードしたバイト列
    return submit_value(prefix, nonce).encode("utf-8")


def sha256_hex(prefix: str, nonce: int) -> str:
    # CPU側で検証する用
    return hashlib.sha256(submit_bytes(prefix, nonce)).hexdigest()


def is_power_of_two(x: int) -> bool:
    return x > 0 and (x & (x - 1)) == 0


def device_name() -> str:
    name = cuda.get_current_device().name
    if isinstance(name, bytes):
        return name.decode()
    return str(name)


def run(
    prefix="mumumu-",
    start=0,
    threads=512,
    blocks=131072,
    device=0,
    progress_sec=10.0,
    max_batches=None,
):
    if threads > MAX_THREADS_PER_BLOCK:
        raise ValueError(f"threads は {MAX_THREADS_PER_BLOCK} 以下にしてね")

    if not is_power_of_two(threads):
        raise ValueError("threads は 2の累乗にしてね。例: 128, 256, 512, 1024")

    if start < 0 or start > UINT64_MAX:
        raise ValueError("start は 0 以上 2^64-1 以下にしてね")

    prefix_bytes = prefix.encode("utf-8")

    # uint64の10進数は最大20桁なので、
    # prefix + 20桁 が 55バイト以下なら1ブロックSHA-256で処理できる
    if len(prefix_bytes) + 20 > 55:
        raise ValueError("prefix は35バイト以下にしてね")

    # 提出値は1〜1024バイトの範囲
    example_len = len(submit_bytes(prefix, start))
    if example_len < 1 or example_len > 1024:
        raise ValueError("提出値の長さが 1〜1024 バイトの範囲外です")

    cuda.select_device(device)

    print("device:", device_name())
    print("prefix:", prefix_bytes)
    print("suffix format: decimal digits")
    print("example submit value:", submit_value(prefix, start))
    print("example bytes hex:", submit_bytes(prefix, start).hex())
    print("example byte length:", example_len)
    print("threads:", threads)
    print("blocks:", blocks)
    print("batch size:", threads * blocks)
    print()

    prefix_np = np.frombuffer(prefix_bytes, dtype=np.uint8).copy()

    d_prefix = cuda.to_device(prefix_np)
    d_K = cuda.to_device(K_HOST)

    d_block_best_scores = cuda.device_array(blocks, dtype=np.int32)
    d_block_best_nonces = cuda.device_array(blocks, dtype=np.uint64)

    print("compiling kernel...")
    mine_max_f_kernel[1, threads](
        d_prefix,
        len(prefix_bytes),
        np.uint64(0),
        d_block_best_scores,
        d_block_best_nonces,
        d_K,
    )
    cuda.synchronize()

    print("start")
    print()

    batch_size = threads * blocks
    next_nonce = start
    checked = 0
    batches = 0

    global_best_score = -1
    global_best_nonce = 0
    global_best_submit = ""
    global_best_hash = ""

    t0 = time.perf_counter()
    last_progress = t0

    while True:
        if max_batches is not None and batches >= max_batches:
            print()
            print("STOP")
            print("checked:", f"{checked:,}")
            print("best:", global_best_score)
            print("best_nonce:", global_best_nonce)
            print("best_submit:", global_best_submit)
            print("best_hash:", global_best_hash)
            return global_best_score, global_best_nonce, global_best_submit, global_best_hash

        if next_nonce > UINT64_MAX - batch_size:
            print("nonce空間を探索しきりました")
            return global_best_score, global_best_nonce, global_best_submit, global_best_hash

        mine_max_f_kernel[blocks, threads](
            d_prefix,
            len(prefix_bytes),
            np.uint64(next_nonce),
            d_block_best_scores,
            d_block_best_nonces,
            d_K,
        )

        scores = d_block_best_scores.copy_to_host()
        nonces = d_block_best_nonces.copy_to_host()

        batch_best_score = int(scores.max())
        candidate_indices = np.flatnonzero(scores == batch_best_score)
        batch_best_nonce = int(nonces[candidate_indices].min())

        checked += batch_size
        next_nonce += batch_size
        batches += 1

        if batch_best_score > global_best_score:
            global_best_score = batch_best_score
            global_best_nonce = batch_best_nonce
            global_best_submit = submit_value(prefix, global_best_nonce)
            global_best_hash = sha256_hex(prefix, global_best_nonce)

            elapsed = time.perf_counter() - t0
            speed = checked / elapsed

            print(
                f"NEW BEST! "
                f"f={global_best_score} "
                f"nonce={global_best_nonce} "
                f"submit={global_best_submit} "
                f"hash={global_best_hash} "
                f"checked={checked:,} "
                f"speed={speed:,.0f} H/s"
            )

        now = time.perf_counter()

        if now - last_progress >= progress_sec:
            elapsed = now - t0
            speed = checked / elapsed

            print(
                f"progress "
                f"checked={checked:,} "
                f"next_nonce={next_nonce:,} "
                f"speed={speed:,.0f} H/s "
                f"current_best_f={global_best_score} "
                f"best_nonce={global_best_nonce} "
                f"best_submit={global_best_submit} "
                f"best_hash={global_best_hash} "
                f"resume=run(prefix={prefix!r}, start={next_nonce}, blocks={blocks}, threads={threads}, progress_sec={progress_sec})"
            )

            last_progress = now

In [17]:
run(
    prefix="mumumu-",
    start=0,
    blocks=2620144,
    threads=1024,
    progress_sec=10
)

device: NVIDIA RTX PRO 6000 Blackwell Server Edition
prefix: b'mumumu-'
suffix format: decimal digits
example submit value: mumumu-0
example bytes hex: 6d756d756d752d30
example byte length: 8
threads: 1024
blocks: 2620144
batch size: 2683027456

compiling kernel...
start

NEW BEST! f=7 nonce=122064758 submit=mumumu-122064758 hash=fffffffd751c1a16a6c59c87d1620d57cede9cf9025e2e8e60d8f7671b6f02b9 checked=2,683,027,456 speed=5,782,783,752 H/s
NEW BEST! f=8 nonce=10570885611 submit=mumumu-10570885611 hash=ffffffffad358d886fad39d5a68a86d8993b9d8e95e431a5f9b14f50a4ad9741 checked=10,732,109,824 speed=5,706,328,220 H/s
progress checked=59,026,604,032 next_nonce=59,026,604,032 speed=5,643,602,783 H/s current_best_f=8 best_nonce=10570885611 best_submit=mumumu-10570885611 best_hash=ffffffffad358d886fad39d5a68a86d8993b9d8e95e431a5f9b14f50a4ad9741 resume=run(prefix='mumumu-', start=59026604032, blocks=2620144, threads=1024, progress_sec=10)
NEW BEST! f=9 nonce=84586447779 submit=mumumu-84586447779 h

KeyboardInterrupt: 